In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow import keras
%matplotlib inline

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def log_loss(y_true, y_predicted):
    if np.any(y_predicted == 0) or np.any(y_predicted == 1):
        return -np.mean(y_true * np.log(np.clip(y_predicted, 1e-15, 1 - 1e-15)) + (1-y_true) * np.log(np.clip(1-y_predicted, 1e-15, 1 - 1e-15)))
    else:
        return -np.mean(y_true * np.log(y_predicted) + (1-y_true) * np.log(1-y_predicted))


In [12]:
class myNN:
    def __init__(self):
        self.w1 = self.w2 = 1
        self.bias = 0
    
    def fit(self, X,y,epochs, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_descent(
            X['age'], X['affordibility'], y, epochs, loss_threshold)

    def predict(self, X_test):
        weighted_sum = self.w1*X_test['age'] + self.w2*X_test['affordibility'] + self.bias
        y_predicted = sigmoid(weighted_sum)
        return y_predicted

    def gradient_descent(self,age, affordibility, y_true, epochs, loss_threshold):
        w1 = w2 = 1
        bias = 0
        learning_rate = 0.1
        n = len(age)

        for i in range(epochs):
            weighted_sum = w1*age + w2*affordibility + bias
            y_predicted = sigmoid(weighted_sum)
            loss = log_loss(y_true, y_predicted)

            dw1 = np.mean(np.dot(np.transpose(age), (y_predicted - y_true)))
            dw2 = np.mean(np.dot(np.transpose(affordibility), (y_predicted - y_true)))
            dbias = np.mean(y_predicted - y_true)

            w1 -= learning_rate * dw1
            w2 -= learning_rate * dw2
            bias -= learning_rate * dbias
            if (i+1) % 50 == 0 or i == 0:
                print(f"Epoch {i+1}: w1={w1}, w2={w2}, bias={bias}, loss:{loss}")
        
            if loss < loss_threshold:
                print(f"Stopping at epoch {i+1} as loss {loss} is below threshold {loss_threshold}")
                break

        return w1, w2, bias

In [13]:
df = pd.read_csv("insurance_data.csv")
df.head()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df[['age', 'affordibility']], df.bought_insurance, test_size=0.2, random_state=42
)
X_train_scale = X_train.copy()
X_train_scale['age'] = X_train_scale['age'] / 100
X_test_scale = X_test.copy()
X_test_scale['age'] = X_test_scale['age'] / 100

In [14]:
custom_model = myNN()
custom_model.fit(X_train_scale, y_train, epochs=5000, loss_threshold=0.4923)

Epoch 1: w1=0.8842357002928037, w2=0.6981107683016998, bias=-0.023497903333540897, loss:0.7428288579142563
Epoch 50: w1=1.4678744393520962, w2=0.2589863191769855, bias=-0.4414492663816414, loss:0.6303814512693239
Epoch 100: w1=2.1785674372852477, w2=0.34232548099562754, bias=-0.8047719562497359, loss:0.5991980728577477
Epoch 150: w1=2.8120741868482515, w2=0.42188998436107444, bias=-1.1325283492233864, loss:0.5739063250784991
Epoch 200: w1=3.3814906332957984, w2=0.4957967117930692, bias=-1.4287149313872032, loss:0.5532844204410842
Epoch 250: w1=3.895079237805556, w2=0.5643556707610065, bias=-1.6972304564729184, loss:0.536361188621155
Epoch 300: w1=4.35972602321461, w2=0.6281235385244629, bias=-1.94153769855608, loss:0.52237500973327
Epoch 350: w1=4.781462864576267, w2=0.6876030255039663, bias=-2.1646344735973986, loss:0.5107323327839969
Epoch 400: w1=5.165559576085452, w2=0.7432176115675082, bias=-2.3690928716802486, loss:0.5009710828910451
Epoch 450: w1=5.516583736398749, w2=0.79532551

In [16]:
y_hat = custom_model.predict(X_test_scale)

In [17]:
y_hat

9     0.833107
25    0.772106
8     0.840664
21    0.244461
0     0.365517
12    0.254831
dtype: float64